# A reusable timeseries annotation template

**Synthetic illustration — no experiment data or movement measurements are used here.**

This notebook demonstrates reusable Matplotlib labels: outbound/inbound regions, derived trial boundaries, raw Start Trial events, decision pokes, trial ends and outcomes. The helper reads no files and can annotate any existing timeseries axis. Movement analysis belongs in the figure notebooks; the artificial signals below only make the labels visible.
Each example separates returned annotation records from rendering. Functions accept
explicit arrays/tables and return their results; only the example calls use
`plt.show()`. This notebook uses Matplotlib and the project's annotation helpers,
not movement measurements.


In [ ]:
from pathlib import Path
import sys

#=== 1| Find the checkout and import reusable annotation functions ========
HERE = Path.cwd().resolve()
PROJECT_SRC = next(
    (
        candidate
        for parent in (HERE, *HERE.parents)
        for candidate in (parent, parent / "src")
        if (candidate / "movement_figures").is_dir()
    ),
    None,
)
if PROJECT_SRC is None:
    raise RuntimeError("Open this notebook from the project checkout or one of its descendants.")
if str(PROJECT_SRC) not in sys.path:
    sys.path.insert(0, str(PROJECT_SRC))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

from movement_figures.timeseries_template.annotations import (
    Annotations,
    Event,
    Region,
    QC_EVENT_STYLES,
    QC_OUTCOME_STYLES,
    QC_REGION_STYLES,
    annotate_qc_timeseries,
    draw_annotations,
    qc_annotations,
)

## Synthetic input in the Q_C table format

All timestamps are **absolute session-clock seconds**. Trials 1 and 2 reached the target zone. Trial 3 has no target trigger and ends as a Miss with `ChosenPort=-1`. Its phase spans will be omitted rather than inventing an inbound/outbound boundary.

The raw Start Trial events deliberately occur after the derived boundaries to show that these are different annotations.

In [ ]:
def make_annotation_example(*, start_time_s: float = 10.0, sample_rate_hz: float = 50.0):
    """Return synthetic trials, events, times and values for the annotation demos.

    Parameters
    ----------
    start_time_s : float
        Absolute time of the first derived trial boundary.
    sample_rate_hz : float
        Sample frequency of the artificial signal; this is not a real recording.

    Returns
    -------
    trials, events, time, values
        Two DataFrames and two matching one-dimensional NumPy arrays. Trial and
        event times are seconds; signal amplitude is in arbitrary units.
    """
    #=== 1| Validate the synthetic clock ========
    if not np.isfinite(start_time_s):
        raise ValueError("start_time_s must be finite.")
    if not np.isfinite(sample_rate_hz) or sample_rate_hz <= 0:
        raise ValueError("sample_rate_hz must be finite and positive.")
    offset = float(start_time_s) - 10.0

    #=== 2| Build three trials with known outcomes and one missing target trigger ========
    trials = pd.DataFrame({
        "session": ["synthetic_session"] * 3,
        "trial_index": [0, 1, 2],
        "start_time": np.array([10.0, 20.0, 30.0]) + offset,
        "end_time": np.array([20.0, 30.0, 40.0]) + offset,
        "outbound_start_time": np.array([10.0, 20.0, 30.0]) + offset,
        "outbound_end_time": np.array([16.0, 26.0, np.nan]) + offset,
        "inbound_start_time": np.array([16.0, 26.0, np.nan]) + offset,
        "inbound_end_time": np.array([20.0, 30.0, 40.0]) + offset,
        "outcome": ["Success", "Failure", "Miss"],
        "ChosenPort": [3, 7, -1],
    })
    events = pd.DataFrame({
        "Time": np.array([10.4, 16.2, 19.8, 20.4, 26.2, 29.8, 30.4]) + offset,
        "Event": [
            "Start Trial 1", "Await poke - Cue Tone ON", "NosePokes & CueTone OFF",
            "Start Trial 2", "Await poke - Cue Tone ON", "NosePokes & CueTone OFF",
            "Start Trial 3",
        ],
        "session": ["synthetic_session"] * 7,
    }).set_index("Time")

    #=== 3| Sample a synthetic signal on the same clock and return every input ========
    sample_count = int(np.floor(32.0 * sample_rate_hz)) + 1
    if sample_count < 2:
        raise ValueError("sample_rate_hz must produce at least two samples over 32 seconds.")
    time = start_time_s - 1.0 + np.arange(sample_count) / sample_rate_hz
    values = 0.5 + 0.25 * np.sin(time * 1.8) + 0.12 * np.cos(time * 4.5)
    return trials, events, time, values


#=== 4| Create the explicitly synthetic inputs for the examples below ========
trials, events, t, signal = make_annotation_example(start_time_s=10.0, sample_rate_hz=50.0)
display(trials)


## Default labels and a key

Each label appears once in the legend even when repeated across trials. Outcome markers sit at 96% of the axes height, independent of the plotted signal's units. A closing event is called a decision poke only when a valid chosen port was recorded and the outcome is not Miss.

In [ ]:
def plot_annotated_trace(
    time,
    values,
    annotations: Annotations,
    *,
    window: tuple[float, float],
    time_offset: float = 0.0,
    ylabel: str = "Amplitude",
    title: str = "Annotated time series",
    ylim: tuple[float, float] | None = None,
    trace_label: str | None = None,
):
    """Return a figure, axis and annotation result for one supplied time series.

    Parameters
    ----------
    time, values : one-dimensional array-like
        Matching samples; time is absolute session seconds. Missing values
        remain gaps. Annotation records use that same absolute clock.
    annotations : Annotations
        Explicit regions and events, typically returned by qc_annotations.
    window : (start, end)
        Absolute interval to display. Subtract time_offset only for presentation.
    time_offset : float
        Origin displayed as zero; does not alter input arrays or annotations.
    ylabel, title, ylim, trace_label
        Axis text, optional limits and optional trace legend label.

    Returns
    -------
    figure, axis, result
        Matplotlib objects and AnnotationResult (artists, legend handles, notes).
        Display or save the returned figure in the calling cell.
    """
    #=== 1| Validate the supplied samples ========
    time = np.asarray(time, dtype=float)
    values = np.asarray(values, dtype=float)
    if time.ndim != 1 or values.shape != time.shape or len(time) < 2:
        raise ValueError("time and values must be matching 1-D arrays with at least two samples.")
    if not np.isfinite(time).all() or not (np.diff(time) > 0).all():
        raise ValueError("time must contain finite increasing seconds.")

    #=== 2| Draw the trace and annotations using the same display offset ========
    fig, ax = plt.subplots(figsize=(13, 4))
    ax.plot(time - time_offset, values, color="0.2", linewidth=1, label=trace_label)
    ax.set(
        xlabel="Session time (s)" if time_offset == 0 else "Time from chosen origin (s)",
        ylabel=ylabel,
        title=title,
    )
    if ylim is not None:
        ax.set_ylim(*ylim)
    result = draw_annotations(
        ax, annotations, window=window, time_offset=time_offset, legend=False,
    )

    #=== 3| Place the key outside the trace and return the figure ========
    if result.legend_handles:
        ax.legend(handles=result.legend_handles, loc="upper center",
                  bbox_to_anchor=(0.5, -0.2), ncol=4, fontsize=8)
    fig.tight_layout()
    return fig, ax, result


#=== 4| Build default annotation records and explicitly display the returned plot ========
default_annotations = qc_annotations(trials=trials, events=events)
fig, ax, result = plot_annotated_trace(
    time=t, values=signal, annotations=default_annotations,
    window=(9.0, 41.0), ylabel="Synthetic amplitude (a.u.)",
    title="Synthetic example: default Q_C annotations", ylim=(-0.05, 1.15),
    trace_label="Synthetic signal",
)
plt.show()
for note in result.notes:
    print(note)


## Select, restyle and extend the template

Passing `None` uses a category's defaults; passing `{}` hides the category. A supplied mapping **replaces** that category, so include each kind you want. Copy an entry before changing it to preserve the shared defaults.

For Q_C regions choose `outbound` and/or `inbound`. Event keys `trial_start`, `poke` and `trial_end` refer to the trial table; other keys match raw event-name prefixes. The generic `Region` and `Event` records add arbitrary intervals and events without changing the Q_C adapter.

In [ ]:
def make_custom_annotations(
    trials,
    events,
    *,
    review_interval: tuple[float, float] = (22.0, 24.0),
    pulse_time: float = 23.0,
    outcome_height: float = 0.90,
) -> Annotations:
    """Return a restyled Q_C annotation set with an explicit review span and pulse.

    Parameters
    ----------
    trials, events : pandas.DataFrame
        One session's trial table and raw events on the same absolute clock.
    review_interval, pulse_time
        Absolute times in seconds for the demonstration-only custom markers.
    outcome_height : float
        Marker height as a fraction of axis height, independent of signal units.

    Returns
    -------
    Annotations
        Immutable record collections to pass to a plotting function. Shared
        default styles and the input trial/event tables are not modified.
    """
    #=== 1| Select categories and copy the default styles before modifying them ========
    base = qc_annotations(
        trials, events,
        region_styles={
            "inbound": {**QC_REGION_STYLES["inbound"], "color": "#c07a32", "alpha": 0.18},
        },
        event_styles={
            "Start Trial": {**QC_EVENT_STYLES["Start Trial"], "color": "#7551a8"},
            "poke": QC_EVENT_STYLES["poke"],
            "trial_end": QC_EVENT_STYLES["trial_end"],
        },
        outcome_styles={
            name: {**style, "height": outcome_height}
            for name, style in QC_OUTCOME_STYLES.items()
        },
    )

    #=== 2| Add custom records and return them with the original adapter notes ========
    return Annotations(
        regions=base.regions + (
            Region(*review_interval, "Example review interval", {"color": "#d5cb46", "alpha": 0.25}),
        ),
        events=base.events + (
            Event(pulse_time, "Example sync pulse", {"color": "#126c7d", "marker": "D"}, height=0.12),
        ),
        notes=base.notes,
    )


#=== 3| Pass the custom result to the same plotting function ========
custom_annotations = make_custom_annotations(
    trials=trials, events=events, review_interval=(22.0, 24.0),
    pulse_time=23.0, outcome_height=0.90,
)
fig, ax, result = plot_annotated_trace(
    time=t, values=signal, annotations=custom_annotations, window=(9.0, 41.0),
    ylabel="Synthetic amplitude (a.u.)", ylim=(-0.05, 1.15),
    title="Synthetic example: selected styles and custom annotations",
    trace_label="Synthetic signal",
)
plt.show()


## Reuse on multiple axes with relative time

Build the records once, then draw on each axis. `window` always uses absolute session time; the displayed x coordinate is `absolute time - time_offset`. Apply the same subtraction to the trace.

Plot windows include both boundaries `[start, end]`, so an entire-trial window includes the closing event and outcome marker. Adjacent panels may therefore each show an event on their shared boundary. This plotting convention does not determine how analysis samples are assigned to trials. Y limits are preserved, including when different panels use different units.

In [ ]:
def plot_annotation_panels(
    time,
    traces: dict[str, np.ndarray],
    annotations: Annotations,
    *,
    window: tuple[float, float],
    time_offset: float = 0.0,
    ylimits: dict | None = None,
    title: str = "Shared time-series annotations",
):
    """Return a multi-panel figure, axes and per-axis annotation results.

    Parameters
    ----------
    time : one-dimensional array-like
        Absolute seconds for every trace.
    traces : dict[str, array-like]
        Ordered mapping from y-axis labels to sampled values.
    annotations : Annotations
        One explicit record set, reused on every panel.
    window, time_offset
        Absolute display bounds and a common displayed time origin in seconds.
    ylimits : dict or None
        Optional mapping from trace label to its (bottom, top) limits.
    title : str
        Title shown above the first panel.

    Returns
    -------
    figure, axes, results
        Figure, one-dimensional axes array, and list of AnnotationResult values.
        Different y scales do not change annotation marker heights or timing.
    """
    #=== 1| Validate every trace before creating axes ========
    time = np.asarray(time, dtype=float)
    if time.ndim != 1 or len(time) < 2 or not np.isfinite(time).all() or not (np.diff(time) > 0).all():
        raise ValueError("time must contain at least two finite increasing seconds.")
    if not traces or any(np.asarray(values).shape != time.shape for values in traces.values()):
        raise ValueError("Provide at least one trace; every trace must match time.")

    #=== 2| Draw each supplied trace with the common time origin ========
    fig, axes_grid = plt.subplots(len(traces), 1, figsize=(13, 3 * len(traces)), sharex=True, squeeze=False)
    axes = axes_grid[:, 0]
    results = []
    handles = {}
    for ax, (label, values) in zip(axes, traces.items()):
        ax.plot(time - time_offset, values, color="0.2", linewidth=1)
        ax.set_ylabel(label)
        if ylimits is not None and label in ylimits:
            ax.set_ylim(*ylimits[label])
        result = draw_annotations(
            ax, annotations, window=window, time_offset=time_offset, legend=False,
        )
        results.append(result)
        for handle in result.legend_handles:
            handles.setdefault(handle.get_label(), handle)

    #=== 3| Add one shared key and return all figure objects ========
    axes[0].set_title(title)
    axes[-1].set_xlabel("Session time (s)" if time_offset == 0 else "Time from chosen origin (s)")
    if handles:
        fig.legend(handles=list(handles.values()), loc="lower center", ncol=4, fontsize=8)
    fig.tight_layout(rect=(0, 0.12, 1, 1))
    return fig, axes, results


#=== 4| Supply two signals and a shared annotation set explicitly ========
shared_annotations = qc_annotations(trials=trials, events=events)
fig, axes, panel_results = plot_annotation_panels(
    time=t,
    traces={"Synthetic amplitude A": signal, "Synthetic amplitude B": 100 + 20 * signal},
    annotations=shared_annotations,
    window=(10.0, 30.0),
    time_offset=float(trials.iloc[0]["start_time"]),
    ylimits={"Synthetic amplitude A": (-0.05, 1.15), "Synthetic amplitude B": (95, 125)},
    title="Synthetic example: shared labels on axes with different scales",
)
plt.show()


## Apply to a real notebook

Load the session once using the import template, select that session's trials and raw events, and plot the movement result first. Then call:

```python
result = annotate_qc_timeseries(
    ax, trials_one_session, events_one_session,
    window=(absolute_start, absolute_end),
)
for note in result.notes:
    print(note)
```

Do not mix sessions or clocks. If raw events are unavailable, pass `events=None`; trial-table labels still work. Missing chosen-port data produces a generic Trial end marker. Missing phase boundaries are reported through `result.notes`. Additional physical beam-break onsets are a separate data source and should be passed as explicit `Event` records when needed.